# Faker Datasets

Description:
- Work with fake datasets

Resources:
- https://github.com/elastic/faker-datasets

Data sources:
- https://raw.githubusercontent.com/dr5hn/countries-states-cities-database/v1.9/cities.json
- https://raw.githubusercontent.com/dr5hn/countries-states-cities-database/v1.9/countries.json

## Example 1:

Generate fake cities

In [ ]:
import urllib.request
import json
from pathlib import Path
from faker import Faker
from faker_datasets import Provider, add_dataset

Path("/tmp/demo_notebooks").mkdir(exist_ok=True)

with urllib.request.urlopen("https://raw.githubusercontent.com/dr5hn/countries-states-cities-database/v1.9/cities.json") as cities:
    data = json.load(cities)
    with open("/tmp/demo_notebooks/cities.json", "w") as file:
        json.dump(data, file)


with urllib.request.urlopen("https://raw.githubusercontent.com/dr5hn/countries-states-cities-database/v1.9/countries.json") as countries:
    data = json.load(countries)
    with open("/tmp/demo_notebooks/countries.json", "w") as file:
        json.dump(data, file)


In [ ]:
@add_dataset("cities", "/tmp/demo_notebooks/cities.json", picker="city")
class Cities(Provider):
    pass


faker = Faker()
faker.add_provider(Cities)

for _ in range(10):
    # Use of the picker named in @add_dataset
    city = faker.city()
    print("{name} is in {country_name}".format(**city))

## Example 2:

Generate fake countries

In [ ]:
from faker_datasets import Provider, add_dataset, with_datasets

@add_dataset("cities", "/tmp/demo_notebooks/cities.json")
class CitiesEx(Provider):

    @with_datasets("cities")
    def city(self, cities):
        return self.__pick__(cities)

faker = Faker()
faker.add_provider(CitiesEx)

for _ in range(10):
    # Use of the picker named in @add_dataset
    city = faker.city()
    print("{name} is in {country_name}".format(**city))

## Example 3:

Generate fake cities and countries

In [ ]:
from faker_datasets import Provider, add_dataset, with_datasets, with_match

@add_dataset("cities", "/tmp/demo_notebooks/cities.json")
@add_dataset("countries", "/tmp/demo_notebooks/countries.json")
class CitiesAndCountries(Provider):

    @with_datasets("cities", "countries")
    def city_by_region(self, cities, countries, region):
        def match(city):
            # Given a city, find its country info in the countries dataset
            country = next(country for country in countries if country["name"] == city["country_name"])
            # Check that the country is in the region of interest
            return country["region"] == region
        return self.__pick__(cities, match=match, max_attempts=10000)


faker = Faker()
faker.add_provider(CitiesAndCountries)

for _ in range(10):
    # Use of the picker named in @add_dataset
    city = faker.city_by_region("Asia")
    print("{name} is in {country_name}".format(**city))